# Sandbox validation notebook

This notebook mirrors dashboard calculations from `app.py` and provides quick checks for key displayed metrics and chart inputs.

In [ ]:
import pandas as pd
import plotly.express as px

import app

hourly_usage, infrastructure_costs, research_requests, users = app.load_datasets_from_zip()

print("Loaded shapes:")
print("hourly_usage:", hourly_usage.shape)
print("infrastructure_costs:", infrastructure_costs.shape)
print("research_requests:", research_requests.shape)
print("users:", users.shape)

In [ ]:
# Product-page top metrics validation
rr = app._lowercase_columns(research_requests).copy()
status_norm = rr["status"].astype(str).str.strip().str.lower()
is_cancelled = app._is_cancelled_status(status_norm)

success_count = int(status_norm.eq("success").sum())
cancelled_count = int(is_cancelled.sum())
total_count = int(len(status_norm))

success_rate = 100.0 * success_count / total_count if total_count else 0.0
cancellation_rate = 100.0 * cancelled_count / total_count if total_count else 0.0

total_request_cost = float(pd.to_numeric(rr["request_cost"], errors="coerce").fillna(0.0).sum())

stream_flag = app._is_true_stream(rr["stream"])
stream_total = int(stream_flag.sum())
non_stream_total = int((~stream_flag).sum())
stream_cancelled = int((is_cancelled & stream_flag).sum())
non_stream_cancelled = int((is_cancelled & ~stream_flag).sum())
stream_cancel_rate = 100.0 * stream_cancelled / stream_total if stream_total else 0.0
non_stream_cancel_rate = 100.0 * non_stream_cancelled / non_stream_total if non_stream_total else 0.0

print({
    "total_request_cost": total_request_cost,
    "success_rate_pct": success_rate,
    "cancellation_rate_pct": cancellation_rate,
    "streaming_cancellation_rate_pct": stream_cancel_rate,
    "non_streaming_cancellation_rate_pct": non_stream_cancel_rate,
})

In [ ]:
# Product-page chart input validation
prepared_cancel = app._prepare_cancellation_chart_data(research_requests)
if prepared_cancel is None:
    print("Cancellation chart data unavailable")
else:
    wait_effect, inefficiency_long, billing_dist, stream_metrics = prepared_cancel
    print("wait_effect")
    display(wait_effect)
    print("inefficiency_long")
    display(inefficiency_long)
    print("billing_dist")
    display(billing_dist)
    print("stream_metrics")
    print(stream_metrics)

q2_data = app._prepare_user_and_cost_breakdowns(users, research_requests)
if q2_data is None:
    print("Q2 chart data unavailable")
else:
    user_dist, request_cost_dist, cost_by_model_user, economics_summary = q2_data
    print("user_dist")
    display(user_dist)
    print("request_cost_dist")
    display(request_cost_dist)
    print("cost_by_model_user")
    display(cost_by_model_user.head(10))
    print("economics_summary")
    print(economics_summary)

In [ ]:
# Infrastructure-page metrics and chart inputs validation
prepared_finops = app._prepare_finops_data(infrastructure_costs, hourly_usage, research_requests)
if prepared_finops is None:
    print("FinOps data unavailable")
else:
    finops_metrics, daily_agg, monthly_agg, heatmap_data = prepared_finops
    print("finops_metrics")
    print(finops_metrics)
    print("daily_agg sample")
    display(daily_agg.head())
    print("monthly_agg")
    display(monthly_agg)
    print("heatmap_data sample")
    display(heatmap_data.head())

    total_cost = finops_metrics["total_hardware_cost"] + finops_metrics["total_ai_cost"]
    infra_share = 100.0 * finops_metrics["total_hardware_cost"] / total_cost if total_cost else 0.0
    model_share = 100.0 * finops_metrics["total_ai_cost"] / total_cost if total_cost else 0.0
    print({"infra_share_pct": infra_share, "model_share_pct": model_share})

    growth_daily = daily_agg.sort_values("day").copy()
    corr_daily = growth_daily[["total_requests", "total_daily_cost"]].dropna()
    pearson_r = float(corr_daily["total_requests"].corr(corr_daily["total_daily_cost"])) if len(corr_daily) > 1 else 0.0
    print({"pearson_r_daily": pearson_r})

In [ ]:
# Weekend-vs-weekday and zero-traffic Research burn validation
hourly_l = app._lowercase_columns(hourly_usage).copy()
infra_l = app._lowercase_columns(infrastructure_costs).copy()

hourly_l["hour"] = pd.to_datetime(hourly_l["hour"], errors="coerce", utc=True)
hourly_l["request_count"] = pd.to_numeric(hourly_l["request_count"], errors="coerce").fillna(0.0)
infra_l["hour"] = pd.to_datetime(infra_l["hour"], errors="coerce", utc=True)

infra_cols = [c for c in infra_l.columns if c.startswith("infra_")]
model_cols = [c for c in infra_l.columns if c.startswith("model_")]
for c in infra_cols + model_cols:
    infra_l[c] = pd.to_numeric(infra_l[c], errors="coerce").fillna(0.0)
infra_l["total_hourly_cost"] = infra_l[infra_cols + model_cols].sum(axis=1)

weekend_req = hourly_l.groupby("hour", as_index=False)["request_count"].sum().rename(columns={"request_count": "total_requests"})
weekend_cost = infra_l.groupby("hour", as_index=False)["total_hourly_cost"].sum()
joined = weekend_req.merge(weekend_cost, on="hour", how="inner").dropna()
joined["day_type"] = joined["hour"].dt.day_name().apply(lambda x: "Weekend" if x in ["Saturday", "Sunday"] else "Weekday")
comp = joined.groupby("day_type", as_index=False).agg(total_requests_mean=("total_requests", "mean"), total_cost_mean=("total_hourly_cost", "mean"))
print("Weekend/Weekday comparison means")
display(comp)

# Research zero-traffic burn
hourly_l["request_type"] = hourly_l["request_type"].astype(str).str.strip().str.lower()
research_hourly = hourly_l.loc[hourly_l["request_type"].eq("research")].groupby("hour", as_index=False)["request_count"].sum().rename(columns={"request_count": "research_requests"})
all_hours = pd.DataFrame({"hour": hourly_l["hour"].drop_duplicates().sort_values().to_list()})
presence = all_hours.merge(research_hourly, on="hour", how="left")
presence["research_requests"] = presence["research_requests"].fillna(0.0)
zero_hours = presence.loc[presence["research_requests"].le(0), "hour"]

if "infra_eks_research_cluster" in infra_l.columns:
    cluster_cost = infra_l[["hour", "infra_eks_research_cluster"]].copy()
    cluster_cost["infra_eks_research_cluster"] = pd.to_numeric(cluster_cost["infra_eks_research_cluster"], errors="coerce").fillna(0.0)
    zero_burn = float(cluster_cost.merge(zero_hours.to_frame(name="hour"), on="hour", how="inner")["infra_eks_research_cluster"].sum())
    print({"zero_traffic_hours": int(len(zero_hours)), "research_cluster_zero_traffic_burn": zero_burn})